[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/082.%20The%20Single%20Facility%20Location%20Problem%20%28Center%20of%20Gravity%29/P82-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 82. The Single Facility Location Problem
## Tier 2: The Classic Heuristic (Geometric Iterative Refinement)

This notebook uses a geometric iterative refinement heuristic to locate one facility serving multiple demand locations. It first computes a demand-weighted center of gravity, generates and evaluates feasible grid candidates near that point, and then applies a local hill-climbing search. The method is a heuristic: it returns a feasible location when one is found, but it does not guarantee a globally optimal solution.

### Goal, inputs, and outputs

- **Goal:** Find a low-cost facility location that satisfies the stated geographic constraints.
- **Inputs:** Demand-location coordinates and weights, a feasible-region rule, a grid resolution and search radius, and a local-search step size.
- **Output:** A feasible location, its total weighted Euclidean distance, and a history of accepted local improvements.
- **Method:** Use the center of gravity as a starting estimate, search nearby grid points, and refine the best feasible grid point by testing eight neighboring directions.

### Model assumptions and scope

Transportation cost is modeled as demand multiplied by straight-line Euclidean distance. The demand values act as weights; the model does not include a separate per-unit transportation rate, fixed facility cost, road-network distance, capacity limits, or uncertainty. Feasibility is determined by the rectangular operating boundary and the restricted zone defined below. The initial candidate set is discretized, while local refinement moves in fixed increments and remains subject to the same feasibility rule.

### Mathematical model

Let there be $m$ demand locations. Location $i$ has coordinates $(a_i,b_i)$ and nonnegative demand weight $w_i$. A facility at $(x,y)$ incurs the following total weighted distance:

$$C(x,y)=\sum_{i=1}^{m} w_i\sqrt{(x-a_i)^2+(y-b_i)^2}.$$

Here, $C(x,y)$ is the objective evaluated by `calculate_weighted_distance`, and the square root gives the Euclidean distance from the facility to demand location $i$. The objective is a weighted sum of distances, not a weighted sum of squared distances. Consequently, the demand-weighted arithmetic center is a useful starting estimate, but it is not generally a minimizer of this objective.

For this example, a location is feasible when it is inside the operating rectangle and outside the restricted zone:

$$\mathcal{F}=\{(x,y):1\leq x\leq12,\ 1\leq y\leq10,\ \text{and not }(x\geq8\text{ and }y\geq4)\}.$$

The feasibility predicate `is_feasible` implements this set. The boundary of the restricted zone is treated as unavailable wherever both inequalities hold.

### Heuristic sequence

1. **Compute the center of gravity.** Use demand weights to obtain an initial estimate:

   $$x_0=\frac{\sum_{i=1}^{m}w_i a_i}{\sum_{i=1}^{m}w_i},\qquad
   y_0=\frac{\sum_{i=1}^{m}w_i b_i}{\sum_{i=1}^{m}w_i}.$$

   In `calculate_center_of_gravity`, these weighted averages provide a convenient starting point. They do not, by themselves, solve the weighted Euclidean-distance minimization problem.

2. **Generate and filter grid candidates.** Examine points in a square centered at $(x_0,y_0)$, with the specified search radius and grid resolution. Keep only points accepted by `is_feasible` (or by the feasibility function supplied to the candidate generator).

3. **Select the best grid candidate.** Evaluate $C(x,y)$ at each feasible candidate and retain a candidate with the smallest cost. If costs tie within numerical tolerance, the first candidate in the generated order is retained.

4. **Refine the location by hill climbing.** From the selected candidate, test the eight displacements north, northeast, east, southeast, south, southwest, west, and northwest. The search accepts the first feasible displacement that strictly reduces cost, then begins another iteration from that location. If no tested displacement improves the cost, it stops; `max_iterations` also limits the search.

If $q_t=(x_t,y_t)$ is the current location, $h$ is the step size, and $D$ is the ordered set of eight direction vectors, the accepted move is the first $d\in D$ for which $q_t+hd\in\mathcal{F}$ and $C(q_t+hd)<C(q_t)$, with numerically equal costs treated as ties. If there is no such direction, the location remains unchanged and the search terminates. The `improvement_history` list records the initial location and each accepted move.

For $G$ grid points and $m$ demand locations, candidate evaluation requires $O(Gm)$ distance calculations. Each local-search iteration tests at most eight moves and requires $O(m)$ distance calculations per move; with $K$ iterations, this part requires $O(Km)$ distance calculations up to the constant factor of eight. These bounds describe the work of this heuristic, not a guarantee of global optimality.

### What to examine

- How feasibility constraints affect which locations can be selected.
- How the grid-based starting point and the local refinement contribute to the final result.
- How the heuristic location and its cost compare with the reference coordinate supplied for Tier 1. The comparison evaluates that coordinate using this notebook's objective; it does not treat the coordinate as a certified optimum.
- How the heuristic behaves when demand data or feasibility rules vary in the robustness scenarios.

### Concrete example: MegaCorp with constraints

The example uses five stores. The restricted environmental zone is $x\geq8$ and $y\geq4$, and the operating boundary is $1\leq x\leq12$ and $1\leq y\leq10$. Grid resolution controls the spacing of initial candidates; local refinement uses a separate step size. The code reports the actual candidate and search results when the notebook is run.

In [ ]:
# These libraries support the heuristic, its data structures, and its visualizations.
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Callable, List, Tuple
from matplotlib.patches import Rectangle

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandLocation:
    """Store a demand point's identifier, coordinates, and demand weight."""

    id: int
    x: float
    y: float
    demand: float

    def distance_to(self, x: float, y: float) -> float:
        """Return the Euclidean distance from this demand point to (x, y)."""
        return math.sqrt((self.x - x) ** 2 + (self.y - y) ** 2)


# The store coordinates and demand weights define the example instance.
stores = [
    DemandLocation(id=1, x=2, y=3, demand=150),
    DemandLocation(id=2, x=8, y=1, demand=200),
    DemandLocation(id=3, x=6, y=7, demand=180),
    DemandLocation(id=4, x=12, y=4, demand=160),
    DemandLocation(id=5, x=4, y=9, demand=110),
]

# This coordinate is retained as a reference point for comparison.
tier1_reference_location = (6.675, 4.425)

print("MegaCorp Store Locations:")
for store in stores:
    print(
        f"Store {store.id}: ({store.x}, {store.y}) "
        f"with demand {store.demand} units"
    )

total_demand = sum(store.demand for store in stores)
print(f"\nTotal Weekly Demand: {total_demand} units")

In [ ]:
def is_feasible(x: float, y: float) -> bool:
    """Check the operating boundary and restricted-zone rule."""
    # The restricted zone is unavailable, including its specified boundary.
    if x >= 8 and y >= 4:
        return False

    # Locations outside the operating rectangle cannot be selected.
    if not (1 <= x <= 12 and 1 <= y <= 10):
        return False

    return True


def calculate_center_of_gravity(
    locations: List[DemandLocation],
) -> Tuple[float, float]:
    """Return the demand-weighted arithmetic center used as a starting estimate."""
    total_demand = sum(location.demand for location in locations)
    if total_demand <= 0:
        raise ValueError("Total demand must be positive to compute the center of gravity.")

    weighted_x = sum(location.x * location.demand for location in locations)
    weighted_y = sum(location.y * location.demand for location in locations)

    center_x = weighted_x / total_demand
    center_y = weighted_y / total_demand

    return center_x, center_y


# Check representative points against the stated geographic rules.
print("Feasibility Testing:")
print("=" * 40)
test_points = [(6.675, 4.425), (8.5, 5), (7, 3), (0.5, 5), (12.5, 8)]
for x, y in test_points:
    feasible = is_feasible(x, y)
    result = "Feasible" if feasible else "Not feasible"
    print(f"Point ({x}, {y}): {result}")

In [ ]:
def generate_grid_candidates(
    center_x: float,
    center_y: float,
    grid_resolution: float = 0.5,
    search_radius: float = 2,
    feasibility_fn: Callable[[float, float], bool] = is_feasible,
) -> List[Tuple[float, float]]:
    """Generate feasible grid points in a square around the starting estimate."""
    if grid_resolution <= 0:
        raise ValueError("Grid resolution must be positive.")
    if search_radius < 0:
        raise ValueError("Search radius must be nonnegative.")

    candidates = []
    x_start = center_x - search_radius
    x_end = center_x + search_radius
    y_start = center_y - search_radius
    y_end = center_y + search_radius

    # Extending each stop by one step includes grid points at the stated range limit.
    x_points = np.arange(x_start, x_end + grid_resolution, grid_resolution)
    y_points = np.arange(y_start, y_end + grid_resolution, grid_resolution)

    print("\nGenerating Grid Candidates:")
    print(f"Center: ({center_x:.3f}, {center_y:.3f})")
    print(f"Grid resolution: {grid_resolution}")
    print(f"Search radius: {search_radius}")
    print(f"X range: [{x_start:.1f}, {x_end:.1f}]")
    print(f"Y range: [{y_start:.1f}, {y_end:.1f}]")

    total_points = 0
    for x in x_points:
        for y in y_points:
            total_points += 1
            if feasibility_fn(float(x), float(y)):
                candidates.append((float(x), float(y)))

    feasible_points = len(candidates)
    feasibility_rate = 100 * feasible_points / total_points if total_points else 0.0
    print(f"Total grid points generated: {total_points}")
    print(f"Feasible candidates: {feasible_points}")
    print(f"Feasibility rate: {feasibility_rate:.1f}%")

    return candidates


# The center is a starting estimate; feasibility filtering determines which nearby points remain.
center_x, center_y = calculate_center_of_gravity(stores)
candidates = generate_grid_candidates(center_x, center_y)

print(f"\nCenter of Gravity: ({center_x:.3f}, {center_y:.3f})")
print(f"Is center feasible? {'Yes' if is_feasible(center_x, center_y) else 'No'}")
print(f"Generated {len(candidates)} feasible candidates")

In [ ]:
def calculate_weighted_distance(
    locations: List[DemandLocation],
    facility_x: float,
    facility_y: float,
) -> float:
    """Evaluate the weighted Euclidean-distance objective at a facility location."""
    total_cost = 0.0
    for location in locations:
        distance = location.distance_to(facility_x, facility_y)
        total_cost += location.demand * distance
    return total_cost


def evaluate_candidates(
    locations: List[DemandLocation],
    candidates: List[Tuple[float, float]],
) -> Tuple[Tuple[float, float], float]:
    """Return the lowest-cost grid candidate and its objective value."""
    if not candidates:
        raise ValueError("At least one feasible candidate is required for evaluation.")

    print("\nEvaluating Candidates:")
    print("=" * 50)

    best_candidate = None
    best_cost = float("inf")

    # A stable candidate order makes the selected point reproducible when costs tie.
    for index, (x, y) in enumerate(candidates):
        cost = calculate_weighted_distance(locations, x, y)
        tied = math.isclose(cost, best_cost, rel_tol=1e-12, abs_tol=1e-9)
        if cost < best_cost and not tied:
            best_cost = cost
            best_candidate = (x, y)

        if index < 5 or (x, y) == best_candidate:
            status = "BEST SO FAR" if (x, y) == best_candidate else ""
            print(f"Candidate ({x:.1f}, {y:.1f}): Cost = {cost:.2f} {status}")

    print(
        f"\nBest grid candidate: ({best_candidate[0]:.3f}, "
        f"{best_candidate[1]:.3f}) with cost {best_cost:.2f}"
    )
    return best_candidate, best_cost


# Evaluate every feasible grid point using the same objective as the local search.
best_grid_candidate, grid_cost = evaluate_candidates(stores, candidates)

In [ ]:
def hill_climbing_improvement(
    locations: List[DemandLocation],
    initial_location: Tuple[float, float],
    step_size: float = 0.1,
    max_iterations: int = 100,
    feasibility_fn: Callable[[float, float], bool] = is_feasible,
) -> Tuple[Tuple[float, float], float, List[Tuple[float, float, float]]]:
    """Refine a feasible starting location by accepting improving moves in eight directions."""
    if step_size <= 0:
        raise ValueError("Step size must be positive.")
    if max_iterations < 1:
        raise ValueError("Maximum iterations must be at least one.")
    if not feasibility_fn(*initial_location):
        raise ValueError("The initial location must be feasible.")

    print("\nLocal Improvement via Hill Climbing:")
    print("=" * 40)

    # The fixed order defines which improving move is selected when several are available.
    directions = [
        (0, step_size),
        (step_size, step_size),
        (step_size, 0),
        (step_size, -step_size),
        (0, -step_size),
        (-step_size, -step_size),
        (-step_size, 0),
        (-step_size, step_size),
    ]

    current_location = [float(initial_location[0]), float(initial_location[1])]
    current_cost = calculate_weighted_distance(
        locations, current_location[0], current_location[1]
    )
    starting_cost = current_cost
    improvement_history = [
        (current_location[0], current_location[1], current_cost)
    ]

    print(f"Starting location: ({current_location[0]:.3f}, {current_location[1]:.3f})")
    print(f"Starting cost: {current_cost:.2f}")

    iteration = 0
    improvements_made = 0

    while iteration < max_iterations:
        iteration += 1
        improved = False

        # First-improvement search moves immediately after finding a lower-cost neighbor.
        for dx, dy in directions:
            new_x = current_location[0] + dx
            new_y = current_location[1] + dy
            if not feasibility_fn(new_x, new_y):
                continue

            new_cost = calculate_weighted_distance(locations, new_x, new_y)
            tied = math.isclose(
                new_cost, current_cost, rel_tol=1e-12, abs_tol=1e-9
            )
            if new_cost < current_cost and not tied:
                current_location = [new_x, new_y]
                current_cost = new_cost
                improvement_history.append(
                    (current_location[0], current_location[1], current_cost)
                )
                improved = True
                improvements_made += 1
                print(
                    f"Iteration {iteration}: Improved to "
                    f"({current_location[0]:.3f}, {current_location[1]:.3f}), "
                    f"cost {current_cost:.2f}"
                )
                break

        if not improved:
            print(f"No improving feasible move found in iteration {iteration}. Stopping.")
            break

    improvement_percent = (
        100 * (starting_cost - current_cost) / starting_cost
        if starting_cost > 0
        else 0.0
    )
    print("\nHill Climbing Results:")
    print(f"Iterations performed: {iteration}")
    print(f"Improvements made: {improvements_made}")
    print(f"Final location: ({current_location[0]:.3f}, {current_location[1]:.3f})")
    print(f"Final cost: {current_cost:.2f}")
    print(f"Improvement from starting point: {improvement_percent:.2f}%")

    return tuple(current_location), current_cost, improvement_history


# Refine the best feasible grid point while retaining the same objective and feasibility rule.
final_location, final_cost, improvement_history = hill_climbing_improvement(
    stores, best_grid_candidate
)

In [ ]:
def compare_with_mathematical_optimum(
    locations: List[DemandLocation],
    reference_location: Tuple[float, float],
    heuristic_location: Tuple[float, float],
    heuristic_cost: float,
) -> None:
    """Compare the heuristic with the supplied Tier 1 reference coordinate."""
    print("\n" + "=" * 60)
    print("COMPARISON WITH THE TIER 1 REFERENCE COORDINATE")
    print("=" * 60)

    reference_cost = calculate_weighted_distance(
        locations, reference_location[0], reference_location[1]
    )
    reference_feasible = is_feasible(*reference_location)

    print(
        f"Reference location: ({reference_location[0]:.3f}, "
        f"{reference_location[1]:.3f})"
    )
    print(f"Reference location cost under this objective: {reference_cost:.2f}")
    print(f"Reference location feasible: {'Yes' if reference_feasible else 'No'}")
    print(
        f"\nHeuristic solution: ({heuristic_location[0]:.3f}, "
        f"{heuristic_location[1]:.3f})"
    )
    print(f"Heuristic solution cost: {heuristic_cost:.2f}")
    print(f"Heuristic solution feasible: {'Yes' if is_feasible(*heuristic_location) else 'No'}")

    if not reference_feasible:
        print("\nThe reference coordinate is outside the feasible region.")
        if reference_cost > 0:
            premium = 100 * (heuristic_cost - reference_cost) / reference_cost
            print(f"Cost difference relative to the reference coordinate: {premium:.2f}%")
    else:
        print(
            "\nBoth locations are feasible. Their costs can be compared directly, "
            "but the reference coordinate is not treated here as a certified optimum."
        )


# Evaluate the reference coordinate with the same data and cost function as the heuristic.
compare_with_mathematical_optimum(
    stores,
    tier1_reference_location,
    final_location,
    final_cost,
)

In [ ]:
def visualize_heuristic_solution() -> None:
    """Plot the feasible region, search history, objective values, and run metrics."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Geometric Iterative Refinement Heuristic",
        fontsize=16,
        fontweight="bold",
    )

    # Panel 1 shows demand points, the reference coordinate, candidates, and the constraints.
    ax1.set_title("Heuristic Solution and Feasibility Constraints", fontweight="bold")
    for store in stores:
        circle_size = store.demand / 8
        ax1.scatter(
            store.x,
            store.y,
            s=circle_size,
            alpha=0.7,
            label=f"Store {store.id}",
            edgecolors="black",
            linewidth=1,
        )
        ax1.annotate(
            f"S{store.id}",
            (store.x, store.y),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=9,
            fontweight="bold",
        )

    reference_x, reference_y = tier1_reference_location
    ax1.scatter(
        reference_x,
        reference_y,
        s=200,
        c="gold",
        marker="o",
        label="Tier 1 reference coordinate",
        edgecolors="black",
        linewidth=2,
    )
    ax1.scatter(
        final_location[0],
        final_location[1],
        s=300,
        c="red",
        marker="*",
        label="Heuristic solution",
        edgecolors="black",
        linewidth=1,
    )

    # The shaded rectangle is exactly the portion of the operating area excluded by the model.
    ax1.add_patch(
        Rectangle(
            (8, 4),
            4,
            6,
            facecolor="red",
            edgecolor="red",
            alpha=0.2,
            label="Restricted area",
        )
    )
    if candidates:
        candidate_x, candidate_y = zip(*candidates)
        ax1.scatter(
            candidate_x,
            candidate_y,
            s=20,
            c="lightblue",
            alpha=0.5,
            label="Feasible grid candidates",
            marker="s",
        )

    ax1.set_xlabel("X coordinate (coordinate units)")
    ax1.set_ylabel("Y coordinate (coordinate units)")
    ax1.set_xlim(1, 12)
    ax1.set_ylim(1, 10)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)

    # Each recorded point corresponds to an accepted move, not every neighbor considered.
    ax2.set_title("Accepted Hill-Climbing Moves", fontweight="bold")
    if improvement_history:
        move_numbers = range(len(improvement_history))
        history_costs = [record[2] for record in improvement_history]
        ax2.plot(move_numbers, history_costs, "b-o", linewidth=2, markersize=5)
        ax2.set_xlabel("Accepted move number")
        ax2.set_ylabel("Total weighted distance (demand × coordinate units)")
        ax2.grid(True, alpha=0.3)
        ax2.annotate(
            f"Start: {history_costs[0]:.1f}",
            (0, history_costs[0]),
            xytext=(8, 8),
            textcoords="offset points",
        )
        ax2.annotate(
            f"Final: {history_costs[-1]:.1f}",
            (len(history_costs) - 1, history_costs[-1]),
            xytext=(-8, -18),
            textcoords="offset points",
        )

    # The reference cost is calculated from the current input data, not hard-coded.
    ax3.set_title("Objective Value Comparison", fontweight="bold")
    reference_cost = calculate_weighted_distance(
        stores, reference_x, reference_y
    )
    comparison_labels = [
        "Tier 1 reference\ncoordinate",
        "Best grid\ncandidate",
        "Final\nheuristic",
    ]
    comparison_costs = [reference_cost, grid_cost, final_cost]
    comparison_colors = ["gold", "lightblue", "red"]
    bars = ax3.bar(
        comparison_labels,
        comparison_costs,
        color=comparison_colors,
        alpha=0.7,
        edgecolor="black",
    )
    ax3.set_ylabel("Total weighted distance (demand × coordinate units)")
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, cost in zip(bars, comparison_costs):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{cost:.0f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    # Work estimates separate candidate count, demand count, and local-search iterations.
    ax4.set_title("Search Summary", fontweight="bold")
    metric_text = (
        "HEURISTIC SEARCH SUMMARY\n"
        "========================\n\n"
        f"Final location: ({final_location[0]:.3f}, {final_location[1]:.3f})\n"
        f"Final objective: {final_cost:.2f}\n"
        f"Feasible: {'Yes' if is_feasible(*final_location) else 'No'}\n\n"
        f"Feasible grid candidates: {len(candidates)}\n"
        f"Best grid objective: {grid_cost:.2f}\n"
        f"Accepted local moves: {len(improvement_history) - 1}\n\n"
        f"Grid evaluation: O(G × m), G={len(candidates)}, m={len(stores)}\n"
        f"Local search: O(K × m), K≤100 iterations\n\n"
        "The heuristic does not guarantee a global optimum."
    )
    ax4.axis("off")
    ax4.text(
        0.05,
        0.95,
        metric_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "lightyellow", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


# Inspect the geographic search and objective values together.
visualize_heuristic_solution()

In [ ]:
def test_robustness_scenarios() -> List[dict]:
    """Run the same heuristic on changed demand data and a separate feasibility rule."""
    print("\n" + "=" * 60)
    print("ROBUSTNESS TESTING UNDER DIFFERENT SCENARIOS")
    print("=" * 60)

    def tight_is_feasible(x: float, y: float) -> bool:
        """Apply the tighter operating boundary and restricted zone for one scenario."""
        return 1 <= x <= 10 and 1 <= y <= 8 and not (x >= 7 and y >= 3)

    scenarios = [
        {
            "name": "Base Case",
            "locations": stores,
            "description": "Original MegaCorp configuration",
            "feasibility_fn": is_feasible,
        },
        {
            "name": "High Demand Store 1",
            "locations": [
                DemandLocation(id=1, x=2, y=3, demand=300),
                DemandLocation(id=2, x=8, y=1, demand=200),
                DemandLocation(id=3, x=6, y=7, demand=180),
                DemandLocation(id=4, x=12, y=4, demand=160),
                DemandLocation(id=5, x=4, y=9, demand=110),
            ],
            "description": "Store 1 demand increased",
            "feasibility_fn": is_feasible,
        },
        {
            "name": "Additional Store",
            "locations": stores + [
                DemandLocation(id=6, x=10, y=8, demand=200)
            ],
            "description": "An additional high-demand store is included",
            "feasibility_fn": is_feasible,
        },
        {
            "name": "Tighter Constraints",
            "locations": stores,
            "description": "Smaller feasible area and a tighter restricted zone",
            "feasibility_fn": tight_is_feasible,
        },
    ]

    results = []
    for scenario in scenarios:
        locations = scenario["locations"]
        feasibility_fn = scenario["feasibility_fn"]
        print(f"\n{scenario['name']}: {scenario['description']}")
        print("-" * 50)

        scenario_center_x, scenario_center_y = calculate_center_of_gravity(locations)
        scenario_candidates = generate_grid_candidates(
            scenario_center_x,
            scenario_center_y,
            feasibility_fn=feasibility_fn,
        )

        if scenario_candidates:
            scenario_best, _ = evaluate_candidates(locations, scenario_candidates)
            scenario_location, scenario_cost, _ = hill_climbing_improvement(
                locations,
                scenario_best,
                feasibility_fn=feasibility_fn,
            )
            results.append(
                {
                    "scenario": scenario["name"],
                    "final_location": scenario_location,
                    "final_cost": scenario_cost,
                    "candidates_generated": len(scenario_candidates),
                    "center_feasible": feasibility_fn(
                        scenario_center_x, scenario_center_y
                    ),
                }
            )
            print(
                f"Solution found: ({scenario_location[0]:.3f}, "
                f"{scenario_location[1]:.3f})"
            )
            print(f"Final cost: {scenario_cost:.2f}")
        else:
            print("No feasible grid candidates were found in the search region.")
            results.append(
                {
                    "scenario": scenario["name"],
                    "final_location": None,
                    "final_cost": float("inf"),
                    "candidates_generated": 0,
                    "center_feasible": feasibility_fn(
                        scenario_center_x, scenario_center_y
                    ),
                }
            )

    print("\n" + "=" * 60)
    print("ROBUSTNESS TESTING SUMMARY")
    print("=" * 60)
    print(
        f"{'Scenario':<22} {'Location':<20} {'Cost':<12} "
        f"{'Candidates':<12} {'Center feasible':<16}"
    )
    print("-" * 82)

    for result in results:
        if result["final_location"] is not None:
            location_text = (
                f"({result['final_location'][0]:.2f}, "
                f"{result['final_location'][1]:.2f})"
            )
            cost_text = f"{result['final_cost']:.2f}"
        else:
            location_text = "No solution"
            cost_text = "N/A"

        center_text = "Yes" if result["center_feasible"] else "No"
        print(
            f"{result['scenario']:<22} {location_text:<20} {cost_text:<12} "
            f"{result['candidates_generated']:<12} {center_text:<16}"
        )

    return results


# Each scenario supplies its own feasibility function, so the base rule remains unchanged.
robustness_results = test_robustness_scenarios()

## Why This Tier Exists Compared with Tier 1

A center-of-gravity calculation provides a useful starting estimate, but practical location problems may include boundaries and restricted areas. This tier incorporates those rules by filtering grid candidates and rejecting infeasible local moves. It retains the weighted-distance objective and adds a transparent search procedure for finding a feasible, lower-cost location.

The grid search explores a finite set near the initial estimate. Hill climbing then makes local, stepwise improvements from the best grid candidate. The two stages have distinct roles: the grid supplies a feasible starting point, while local search can move beyond the grid points at the initial resolution. Neither stage proves that the returned location is globally optimal.

## Advantages and limitations

**Advantages**

- Feasibility rules are explicit and can be inspected independently of the objective calculation.
- The search and accepted improvements are visible and interpretable.
- Demand data and feasibility rules can be varied to examine how the heuristic responds.
- The grid and local-search parameters expose the trade-off between search detail and computational work.

**Limitations**

- The heuristic has no general guarantee of finding a global optimum.
- The result depends on the search region, grid resolution, step size, move order, and starting candidate.
- A local stopping point has no improving move among the eight tested directions at the chosen step size; it need not be optimal over all feasible coordinates.
- The weighted Euclidean-distance model omits road networks, fixed costs, capacity, and other operational factors not specified in this example.

## When to use this tier

This approach is useful for illustrating how geographic feasibility rules can be incorporated into a transparent facility-location heuristic. It is appropriate when a practical, interpretable candidate is valuable and the search settings are suitable for the scale of the example. For any application, verify that the distance measure, demand weights, feasible region, and location resolution represent the decision being modeled.

## Key takeaways

The method combines a demand-weighted starting estimate, feasible grid enumeration, objective evaluation, and first-improvement hill climbing in eight directions. The cost function remains the weighted sum of Euclidean distances throughout. Feasibility is enforced both when constructing the initial candidate set and when considering local moves. The reported location is therefore a heuristic result for the stated data and parameters, not a proof of global optimality.